# Eye Tracking Session Analysis

Offline analysis of a recorded session CSV. Covers:
- Gaze trajectory and spatial distribution
- Blink detection via Eye Aspect Ratio (EAR)
- Fixation timeline and rolling rate
- Head pose (yaw)
- Area-of-Interest dwell time
- Gaze heatmap
- Gaze direction (if captured with `--export-ply` pipeline)

Set `CSV_PATH` below to point at any session file, or leave it as `None` to auto-load the most recent one.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# --- configure ---
DATA_DIR = Path('../data')
CSV_PATH = None   # set to a specific path, or None to use the most recent

if CSV_PATH is None:
    candidates = sorted(DATA_DIR.glob('gaze_*.csv'))
    if not candidates:
        raise FileNotFoundError(f'No gaze_*.csv files found in {DATA_DIR}')
    CSV_PATH = candidates[-1]

print(f'Loading: {CSV_PATH}')

In [ ]:
df = pd.read_csv(CSV_PATH)

# Relative time in seconds from session start
df['time_s'] = df['timestamp'] - df['timestamp'].iloc[0]

duration    = df['time_s'].max()
fps_est     = len(df) / max(duration, 1e-6)
face_pct    = df['gaze_x'].notna().mean() * 100
n_blinks    = int(df['is_blink'].sum())
fix_pct     = df['is_fixation'].mean() * 100

print(f'Session  : {CSV_PATH.name}')
print(f'Duration : {duration:.1f} s   |   Frames: {len(df)}   |   Est. FPS: {fps_est:.1f}')
print(f'Face detected  : {face_pct:.1f}%')
print(f'Blinks         : {n_blinks}')
print(f'Fixation frames: {df["is_fixation"].sum()}  ({fix_pct:.1f}%)')

## Gaze Trajectory

In [ ]:
valid = df.dropna(subset=['gaze_x', 'gaze_y'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spatial scatter coloured by time
sc = axes[0].scatter(
    valid['gaze_x'], valid['gaze_y'],
    c=valid['time_s'], cmap='viridis', s=4, alpha=0.6
)
axes[0].invert_yaxis()   # image coords: y increases downward
axes[0].set_xlabel('Gaze X (px)')
axes[0].set_ylabel('Gaze Y (px)')
axes[0].set_title('Gaze Scatter  (colour = time)')
plt.colorbar(sc, ax=axes[0], label='Time (s)')

# Time series
axes[1].plot(valid['time_s'], valid['gaze_x'], label='X', alpha=0.75, lw=0.9)
axes[1].plot(valid['time_s'], valid['gaze_y'], label='Y', alpha=0.75, lw=0.9)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Position (px)')
axes[1].set_title('Gaze Position Over Time')
axes[1].legend()

plt.tight_layout()
plt.show()

## Blink Detection — Eye Aspect Ratio (EAR)

EAR = (vertical eye opening) / (horizontal eye width).  
A blink is flagged when the mean EAR drops below **0.20**.

In [ ]:
valid_ear = df.dropna(subset=['left_ear', 'right_ear'])
avg_ear   = (valid_ear['left_ear'] + valid_ear['right_ear']) / 2

fig, ax = plt.subplots(figsize=(14, 4))

ax.plot(valid_ear['time_s'], valid_ear['left_ear'],  label='Left EAR',  alpha=0.55, lw=0.8)
ax.plot(valid_ear['time_s'], valid_ear['right_ear'], label='Right EAR', alpha=0.55, lw=0.8)
ax.plot(valid_ear['time_s'], avg_ear,                label='Mean EAR',  color='k',  lw=1.2)
ax.axhline(0.20, color='red', ls='--', lw=1.0, label='Blink threshold (0.20)')

for _, row in df[df['is_blink']].iterrows():
    ax.axvline(row['time_s'], color='red', alpha=0.35, lw=1.5)

ax.set_xlabel('Time (s)')
ax.set_ylabel('EAR')
ax.set_title(f'Eye Aspect Ratio — {n_blinks} blink(s) detected')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Fixation Analysis

Fixations are classified using the **I-VT** (velocity threshold) algorithm:  
gaze velocity < 25 px/s sustained for ≥ 100 ms.

In [ ]:
window = max(1, int(fps_est))   # 1-second rolling window
rolling_fix = df['is_fixation'].rolling(window, center=True, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].fill_between(df['time_s'], df['is_fixation'].astype(int),
                     alpha=0.45, color='steelblue')
axes[0].set_xlabel('Time (s)')
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Saccade', 'Fixation'])
axes[0].set_title('Fixation / Saccade Timeline')

axes[1].plot(df['time_s'], rolling_fix, color='steelblue', lw=1.2)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Fixation rate')
axes[1].set_ylim(0, 1)
axes[1].set_title(f'Rolling Fixation Rate  ({window}-frame window)')

plt.tight_layout()
plt.show()

print(f'Overall fixation rate: {fix_pct:.1f}%')

## Head Pose — Yaw

Head pose is recovered via `cv2.solvePnP` on 6 facial landmarks.  
Yaw (left/right rotation) is the most stable of the three Euler angles;  
pitch and roll can exhibit wrap-around artefacts from `RQDecomp3x3`.

In [ ]:
valid_pose = df.dropna(subset=['yaw'])

# Clip to plausible range to suppress wrap-around artefacts
yaw_clipped = valid_pose['yaw'].clip(-60, 60)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(valid_pose['time_s'], yaw_clipped, color='darkorange', lw=0.9)
ax.fill_between(valid_pose['time_s'], yaw_clipped, alpha=0.18, color='darkorange')
ax.axhline(0, color='grey', ls='--', lw=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Yaw (°)')
ax.set_title('Head Yaw Over Time  (+ = turned right, − = turned left)')
plt.tight_layout()
plt.show()

print(f'Mean yaw: {valid_pose["yaw"].mean():.1f}°  |  Std: {valid_pose["yaw"].std():.1f}°')

## Area-of-Interest (AOI) Dwell Time

In [ ]:
aoi_counts = df['active_aoi'].value_counts()

if aoi_counts.empty:
    print('No AOI data in this session.')
else:
    aoi_seconds = aoi_counts / fps_est
    colors = plt.cm.Set2(np.linspace(0, 1, len(aoi_seconds)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    bars = axes[0].bar(aoi_seconds.index, aoi_seconds.values, color=colors)
    for bar, val in zip(bars, aoi_seconds.values):
        axes[0].text(bar.get_x() + bar.get_width() / 2,
                     bar.get_height() + 0.02,
                     f'{val:.1f}s', ha='center', va='bottom')
    axes[0].set_ylabel('Dwell time (s)')
    axes[0].set_title('AOI Dwell Time')

    axes[1].pie(aoi_seconds.values, labels=aoi_seconds.index,
                colors=colors, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('AOI Share')

    plt.tight_layout()
    plt.show()

## Gaze Heatmap

2D histogram of gaze positions, smoothed with a Gaussian kernel.

In [ ]:
valid_gaze = df.dropna(subset=['gaze_x', 'gaze_y'])

# Build accumulator at frame resolution, then smooth
frame_h, frame_w = 480, 640   # adjust if your recording used a different resolution
heat = np.zeros((frame_h, frame_w), dtype=np.float32)
for _, row in valid_gaze.iterrows():
    x, y = int(row['gaze_x']), int(row['gaze_y'])
    if 0 <= x < frame_w and 0 <= y < frame_h:
        heat[y, x] += 1.0

heat_blur = gaussian_filter(heat, sigma=18)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heat_blur, cmap='jet', origin='upper', aspect='equal')
plt.colorbar(im, ax=ax, label='Gaze density (smoothed)')
ax.set_title('Gaze Heatmap')
ax.set_xlabel('X (px)')
ax.set_ylabel('Y (px)')
plt.tight_layout()
plt.show()

## Gaze Direction (iris + head pose fused)

Available in sessions recorded with the current pipeline.  
`dir_h` / `dir_v` are in \[−1, 1\]: 0 = straight ahead.

In [ ]:
if 'dir_h' not in df.columns or df['dir_h'].isna().all():
    print('No gaze direction data in this session.\n'
          'Re-record with the current pipeline to capture dir_h / dir_v.')
else:
    valid_dir = df.dropna(subset=['dir_h', 'dir_v'])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: direction distribution
    sc = axes[0].scatter(
        valid_dir['dir_h'], valid_dir['dir_v'],
        c=valid_dir['time_s'], cmap='viridis', s=5, alpha=0.6
    )
    axes[0].set_xlim(-1, 1)
    axes[0].set_ylim(-1, 1)
    axes[0].axhline(0, color='grey', ls='--', lw=0.6)
    axes[0].axvline(0, color='grey', ls='--', lw=0.6)
    axes[0].set_xlabel('dir_h  (−1 = left, +1 = right)')
    axes[0].set_ylabel('dir_v  (−1 = up, +1 = down)')
    axes[0].set_title('Gaze Direction Distribution')
    plt.colorbar(sc, ax=axes[0], label='Time (s)')

    # Time series
    axes[1].plot(valid_dir['time_s'], valid_dir['dir_h'], label='Horizontal', lw=0.9, alpha=0.8)
    axes[1].plot(valid_dir['time_s'], valid_dir['dir_v'], label='Vertical',   lw=0.9, alpha=0.8)
    axes[1].axhline(0, color='grey', ls='--', lw=0.6)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Direction')
    axes[1].set_title('Gaze Direction Over Time')
    axes[1].legend()

    plt.tight_layout()
    plt.show()